In [ ]:
import os
# Prevent JAX from pre-allocating all GPU memory
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["JAX_DISABLE_MMAP_CACHE"] = "1"
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=2"

import glob, time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update('jax_enable_x64', True)
jax.clear_caches()
import jax.numpy as jnp

import discovery as ds
from enterprise_extensions import load_feathers
from discovery.deterministic import make_phase_connected_binary
from discovery import const as disco_const
from discovery.deterministic import fpcmu_fast

import scipy.linalg as sl
import scipy.sparse as ss
from sksparse.cholmod import cholesky
from enterprise.signals import signal_base
from enterprise.signals.gp_signals import get_timing_model_basis, BasisGP
from enterprise.signals.parameter import function
from enterprise.signals import white_signals, gp_signals, utils, selections, parameter
from enterprise_extensions.blocks import common_red_noise_block
from enterprise_extensions.deterministic import cw_block_circ

print('Imports OK')

In [ ]:
# =============================================================================
# CONFIGURABLE PARAMETERS — MULTI-CW, MULTI-PULSAR

In [ ]:
# =============================================================================
Npulsars = 5       # Number of pulsars to include
N_CW = 3           # Number of CW sources
NCW8 = 8 * N_CW    # Number of deterministic block parameters (total)
log10_h = -13.0    # log10(strain amplitude) for all sources, -12 = high SNR, -13 = moderate

# Toggle: whether to sample red noise / GWB hyperparameters or fix at injection
SAMPLE_NOISE = False

# Sampler schedule
n_anneal = 15000
n_adapt  = 5000
n_prod   = 5000
T_start  = 5000.0
T_end    = 1.0

# Noise injection parameters
INJECT_REDNOISE = True
INJECT_GWB      = False

KPC_OVER_C = disco_const.kpc / disco_const.c  # kpc -> light-seconds

print(f"Configuration: N_CW={N_CW}, NCW8={NCW8}, Npulsars={Npulsars}, h=10^{log10_h}")
print(f"SAMPLE_NOISE={SAMPLE_NOISE}, INJECT_REDNOISE={INJECT_REDNOISE}, INJECT_GWB={INJECT_GWB}")

feather_dir = "../data_products/"

In [ ]:
# =============================================================================
# LOAD PULSARS — both discovery and enterprise versions

In [ ]:
# =============================================================================
disco_psrs = [ds.Pulsar.read_feather(f) for f in sorted(glob.glob(feather_dir + "*.feather"))][:Npulsars]
print(f"Loaded {len(disco_psrs)} pulsars: {[p.name for p in disco_psrs]}")

# Enterprise pulsars (needed for simulate() and distance priors)
psrs_ent = load_feathers.load_feathers_from_folder(feather_dir)
ent_by_name = {p.name: p for p in psrs_ent}
psrs_ent_used = [ent_by_name[p.name] for p in disco_psrs]

# Load EM distance priors
# These will be used as EM-centric Gaussians for distance prior and for injection offset
dist_mu, dist_sig = [], []
for psr in disco_psrs:
    ep  = ent_by_name[psr.name]
    mu  = float(ep.pdist[0])
    sig = float(ep.pdist[1]) if len(ep.pdist) > 1 else 0.5
    if (not np.isfinite(sig)) or sig <= 0:
        sig = 0.5
    dist_mu.append(mu)
    dist_sig.append(sig)

print(f"dist_mu:  {[f'{m:.3f}' for m in dist_mu]}")
print(f"dist_sig: {[f'{s:.4f}' for s in dist_sig]}")

# JAX/numpy arrays used throughout
mu_arr = np.array(dist_mu)
sd_arr = np.array(dist_sig)
dist_mu_jnp = jnp.array(dist_mu, dtype=jnp.float64)
sd_jnp      = jnp.array(dist_sig, dtype=jnp.float64)
psr_toas_list = [np.asarray(psr.toas, dtype=np.float64) for psr in disco_psrs]
psr_pos_list  = [psr.pos for psr in disco_psrs]
psr_positions = jnp.array([psr.pos for psr in disco_psrs])

In [ ]:
# =============================================================================
# MULTI-SOURCE INJECTION PARAMETERS

In [ ]:
# =============================================================================
CW_PARAM_NAMES = ['cos_gwtheta', 'gwphi', 'cos_inc', 'log10_mc',
                  'log10_fgw', 'log10_h', 'phase0', 'psi']

rng_inj = np.random.default_rng(12345)
INJ_LIST = [
    # Source 0: fixed
    {"cos_gwtheta": 0.3, "gwphi": 2.5, "cos_inc": -0.2,
     "log10_mc": 9.0, "log10_fgw": -8.0, "log10_h": log10_h,
     "phase0": 1.0, "psi": 0.7},
]
for s in range(1, N_CW):
    INJ_LIST.append({
        "cos_gwtheta": float(rng_inj.uniform(-1, 1)),
        "gwphi":       float(rng_inj.uniform(0, 2*np.pi)),
        "cos_inc":     float(rng_inj.uniform(-1, 1)),
        "log10_mc":    float(rng_inj.uniform(8.5, 9.5)),
        "log10_fgw":   float(rng_inj.uniform(-8.5, -7.5)),
        "log10_h":     log10_h,
        "phase0":      float(rng_inj.uniform(0, 2*np.pi)),
        "psi":         float(rng_inj.uniform(0, np.pi)),
    })

DIST_OFFSET_SIGMA = 0.3
true_dists = {psr.name: float(dist_mu[i]) + DIST_OFFSET_SIGMA * float(dist_sig[i])
              for i, psr in enumerate(disco_psrs)}

# Map to 'discovery' parameter naming: each param key as cw_param, cw_param_2, ...
CW_DISCO_NAMES = []
for idx in range(N_CW):
    suffix = "" if idx == 0 else f"_{idx+1}"
    for key in CW_PARAM_NAMES:
        CW_DISCO_NAMES.append(f"cw_{key}{suffix}")
N_CW_PARAMS = len(CW_DISCO_NAMES)  # == NCW8
DIST_DISCO_NAMES = [f"{psr.name}_cw_p_dist" for psr in disco_psrs]

# Build injection dict in 'discovery' convention
inj_cw_vals = {}
for idx, inj in enumerate(INJ_LIST):
    suffix = "" if idx == 0 else f"_{idx+1}"
    for key in CW_PARAM_NAMES:
        inj_cw_vals[f"cw_{key}{suffix}"] = inj[key]
for i, psr in enumerate(disco_psrs):
    inj_cw_vals[f"{psr.name}_cw_p_dist"] = true_dists[psr.name]

In [ ]:
# =============================================================================
# ENTERPRISE PTA MODEL — simulation with N_CW sources

In [ ]:
# =============================================================================
# Enterprise timing model with improper prior — needed for simulate() to include timing model basis in the GP covariance.
# The variance=1e40 is intentionally huge (improper prior).
@function
def tm_prior(weights, toas, variance=1e40):
    return weights * variance * len(toas)

def TimingModel(coefficients=False, name="linear_timing_model",
                use_svd=False, normed=True, prior_variance=1e40):
    basis = get_timing_model_basis(use_svd, normed)
    prior = tm_prior(variance=prior_variance)
    BaseClass = BasisGP(prior, basis, coefficients=coefficients, name=name)
    class TimingModel(BaseClass):
        signal_type = "basis"
        signal_name = "linear timing model"
        signal_id = name + "_svd" if use_svd else name
    return TimingModel

print("Building enterprise PTA model for data simulation...")
for psr in psrs_ent_used:
    psr._pdist = psr.pdist
    psr.residuals = np.array(psr.toas) * 0.0

tmin = [p.toas.min() for p in psrs_ent_used]
tmax = [p.toas.max() for p in psrs_ent_used]
Tspan = np.max(tmax) - np.min(tmin)

selection = selections.Selection(selections.by_backend)
ef = parameter.Constant(1)
eq = parameter.Constant(-8)

log10_A_red = parameter.Uniform(-18, -11)
gamma_red   = parameter.Uniform(0, 7)

log10_A_gw = parameter.Uniform(-18, -11)('gwb_log10_A')
gamma_gw   = parameter.Uniform(0, 7)('gwb_gamma')

components = 30

tm = TimingModel(coefficients=False, name="linear_timing_model", use_svd=False, normed=True, prior_variance=1e-14)

models = []
for p in psrs_ent_used:
    s = tm
    efac  = white_signals.MeasurementNoise(efac=ef, selection=selection)
    s += efac
    equad = white_signals.TNEquadNoise(log10_tnequad=eq, selection=selection)
    s += equad
    pl = utils.powerlaw(log10_A=log10_A_red, gamma=gamma_red)
    rn = gp_signals.FourierBasisGP(spectrum=pl, components=components, Tspan=Tspan, name="rednoise")
    s += rn
    crn = common_red_noise_block(psd='powerlaw', prior='log-uniform', components=components, orf='hd', name='gwb')
    s += crn
    # Add N_CW deterministic blocks, one per source, with unique names (cw, cw_2, ...)
    for idx in range(N_CW):
        suffix = "" if idx == 0 else f"_{idx+1}"
        s += cw_block_circ(
            amp_prior="log-uniform", dist_prior=None, skyloc=None,
            log10_fgw=None, psrTerm=True, phase_connected=True,
            discoclone=False, name=f"cw{suffix}",
        )
    models.append(s(p))

pta = signal_base.PTA(models)
pta.set_default_params({})
print(f"Enterprise PTA model built with {len(pta.pulsars)} pulsars, {N_CW} CW blocks")

In [ ]:
# =============================================================================
# ENTERPRISE PARAMETER DICTIONARY FOR SIMULATION

In [ ]:
# =============================================================================
np.random.seed(42)
enterprise_params = {}
if INJECT_REDNOISE:
    for p in pta.param_names:
        if 'rednoise_log10_A' in p:
            enterprise_params[p] = np.random.uniform(-16, -13)
        elif 'rednoise_gamma' in p:
            enterprise_params[p] = np.random.uniform(2, 6)
else:
    for p in pta.param_names:
        if 'rednoise_log10_A' in p:
            enterprise_params[p] = -18.0
        elif 'rednoise_gamma' in p:
            enterprise_params[p] = 3.0
if INJECT_GWB:
    enterprise_params['gwb_gamma']   = 4.333
    enterprise_params['gwb_log10_A'] = -14.5
else:
    enterprise_params['gwb_gamma']   = 4.333
    enterprise_params['gwb_log10_A'] = -18.0
for psr in psrs_ent_used:
    for p in pta.param_names:
        if psr.name + '_cw_p_dist' in p:
            enterprise_params[p] = true_dists[psr.name]

# CW parameters: use enterprise naming convention
# Enterprise block names: cw, cw_2, cw_3 → params: cw_cos_gwtheta, cw_2_cos_gwtheta, etc.
cw_block_names = ["cw" if idx == 0 else f"cw_{idx+1}" for idx in range(N_CW)]
for idx, (inj, block_name) in enumerate(zip(INJ_LIST, cw_block_names)):
    for key in CW_PARAM_NAMES:
        ent_key = f"{block_name}_{key}"
        enterprise_params[ent_key] = inj[key]

print("\nInjection parameters:")
for k, v in sorted(enterprise_params.items()):
    print(f"  {k}: {v}")

In [ ]:
# =============================================================================
# SIMULATE DATA (CW + NOISE + RED NOISE + GWB)

In [ ]:
# =============================================================================
def simulate(pta, params, sparse_cholesky=True):
    """
    enterprise's simulate() generates fake residuals by:
      (1) computing deterministic delays (CW signals),
      (2) drawing a GP noise realization from the full covariance (red noise + GWB via Cholesky of the joint Fourier basis phi matrix),
      and (3) drawing white noise.
    The result is delay + GP_noise + white_noise per pulsar.
    """
    delays, ndiags, fmats, phis = (pta.get_delay(params=params),
                                   pta.get_ndiag(params=params),
                                   pta.get_basis(params=params),
                                   pta.get_phi(params=params))
    gpresiduals = []
    if pta._commonsignals:
        if sparse_cholesky:
            cf = cholesky(ss.csc_matrix(phis))
            gp = np.zeros(phis.shape[0])
            gp[cf.P()] = np.dot(cf.L().toarray(), np.random.randn(phis.shape[0]))
        else:
            gp = np.dot(sl.cholesky(phis, lower=True), np.random.randn(phis.shape[0]))
        i = 0
        for fmat in fmats:
            j = i + fmat.shape[1]
            gpresiduals.append(np.dot(fmat, gp[i:j]))
            i = j
        assert len(gp) == i
    else:
        for fmat, phi in zip(fmats, phis):
            if phi is None:
                gpresiduals.append(0)
            elif phi.ndim == 1:
                gpresiduals.append(np.dot(fmat, np.sqrt(phi) * np.random.randn(phi.shape[0])))
            else:
                raise NotImplementedError
    whiteresiduals = []
    for delay, ndiag in zip(delays, ndiags):
        if ndiag is None:
            whiteresiduals.append(0)
        elif isinstance(ndiag, signal_base.ShermanMorrison):
            n = np.diag(ndiag._nvec)
            for j, s in zip(ndiag._jvec, ndiag._slices):
                n[s, s] += j
            whiteresiduals.append(delay + np.dot(sl.cholesky(n, lower=True), np.random.randn(n.shape[0])))
        elif ndiag.ndim == 1:
            whiteresiduals.append(delay + np.sqrt(ndiag) * np.random.randn(ndiag.shape[0]))
        else:
            raise NotImplementedError
    return [np.array(g + w) for g, w in zip(gpresiduals, whiteresiduals)]

print("\nSimulating data with enterprise (including all CW sources)...")
sim_resids = simulate(pta, enterprise_params, sparse_cholesky=True)
name_to_resid = {getattr(p, "name", p): y for p, y in zip(pta.pulsars, sim_resids)}
data_list = [name_to_resid[psr.name] for psr in disco_psrs]
print(f"Simulated residuals std: {[f'{np.std(d):.2e}' for d in data_list]}")

In [ ]:
# =============================================================================
# DISCOVERY: MULTI-SOURCE CW DELAY CALLABLE

In [ ]:
# =============================================================================
class MultiSourceDelay:
    """
    Wraps N CW sources into a single delay callable for discovery's PulsarLikelihood. Sums the delay contributions from each source. The `.params` attribute tells discovery which dict keys this callable depends on, so it can trace parameter dependencies. Discovery naming: cw_param (source 0), cw_param_2 (source 1), cw_param_3 (source 2), etc.
    """
    """Wraps CW delay for N sources. Discovery naming: cw_param, cw_param_2, etc."""
    global_params = ("cos_gwtheta", "gwphi", "cos_inc", "log10_mc", "log10_fgw", "log10_h", "phase0", "psi")
    def __init__(self, psr, n_sources):
        self.psr = psr
        self.n_sources = n_sources
        self.single_source = make_phase_connected_binary(pulsarterm=True)
        params = []
        for idx in range(n_sources):
            suffix = "" if idx == 0 else f"_{idx+1}"
            for key in self.global_params:
                name = f"cw_{key}{suffix}"
                if name not in params:
                    params.append(name)
        params.append(f"{self.psr.name}_cw_p_dist")
        self.params = params
    def __call__(self, params):
        total = 0.0
        for idx in range(self.n_sources):
            suffix = "" if idx == 0 else f"_{idx+1}"
            kwargs = {key: params[f"cw_{key}{suffix}"] for key in self.global_params}
            kwargs["p_dist"] = params[f"{self.psr.name}_cw_p_dist"]
            total += self.single_source(self.psr.toas, self.psr.pos, **kwargs)
        return total

# Build PulsarLikelihood for each pulsar
# Use the MultiSourceDelay instead of SingleSourceDelay
noisedict = {}
noisedict.update({psr.name + "_KAT_MKBF_efac": 1.0 for psr in disco_psrs})
noisedict.update({psr.name + "_KAT_MKBF_log10_t2equad": -8.0 for psr in disco_psrs})
noise_terms = {psr.name: ds.makenoise_measurement(psr, noisedict=noisedict) for psr in disco_psrs}
timing_terms = {psr.name: ds.makegp_timing(psr, variance=1e-14) for psr in disco_psrs}
T_disco = ds.getspan(disco_psrs)
common_gp = ds.makecommongp_fourier(disco_psrs, ds.powerlaw, 30, T_disco, name="rednoise")
global_gp = ds.makeglobalgp_fourier(disco_psrs, ds.powerlaw, ds.hd_orf, 30, T_disco, name="gwb")

pulsar_likes = [
    ds.PulsarLikelihood([
        np.array(data_list[i], copy=True),
        noise_terms[psr.name],
        timing_terms[psr.name],
        MultiSourceDelay(psr, N_CW),
    ])
    for i, psr in enumerate(disco_psrs)
]

fml = ds.ArrayLikelihood(pulsar_likes, commongp=common_gp, globalgp=global_gp)
logl_disco = fml.logL
print(f"Discovery likelihood built. Parameters: {logl_disco.params}")

In [ ]:
# =============================================================================
# PARAMETER MAPPING AND BOUNDS

In [ ]:
# =============================================================================
# Red noise and GWB discovery names if needed
RN_DISCO_NAMES = []
for psr in disco_psrs:
    RN_DISCO_NAMES.append(f"{psr.name}_rednoise_log10_A")
    RN_DISCO_NAMES.append(f"{psr.name}_rednoise_gamma")
GWB_DISCO_NAMES = ["gwb_log10_A", "gwb_gamma"]

if SAMPLE_NOISE:
    SAMPLED_NAMES = CW_DISCO_NAMES + DIST_DISCO_NAMES + RN_DISCO_NAMES + GWB_DISCO_NAMES
else:
    SAMPLED_NAMES = CW_DISCO_NAMES + DIST_DISCO_NAMES

Ndim = len(SAMPLED_NAMES)

print(f"\nSampled parameters ({Ndim} total):")
for i, name in enumerate(SAMPLED_NAMES):
    print(f"  [{i:2d}] {name}")

# Fixed parameters: what's needed by logl_disco but not sampled
FIXED_PARAMS = {}
for p in logl_disco.params:
    if p not in SAMPLED_NAMES:
        if p in inj_cw_vals:
            FIXED_PARAMS[p] = inj_cw_vals[p]
        elif p in enterprise_params:
            FIXED_PARAMS[p] = enterprise_params[p]
        else:
            print(f"  WARNING: parameter '{p}' required but not found in injection dict!")

# Parameter bounds
CW_BOUNDS_LO = np.array([-1.0, 0.0,        -1.0, 7.0, -9.0, -18.0, 0.0,        0.0])
CW_BOUNDS_HI = np.array([ 1.0, 2*np.pi,     1.0, 10.0, -7.0, -11.0, 2*np.pi,   np.pi])
if SAMPLE_NOISE:
    RN_BOUNDS_LO = np.tile([-18.0, 0.0], Npulsars)
    RN_BOUNDS_HI = np.tile([-11.0, 7.0], Npulsars)
    GWB_BOUNDS_LO = np.array([-18.0, 0.0])
    GWB_BOUNDS_HI = np.array([-11.0, 7.0])
    PARAM_LO = np.concatenate([np.tile(CW_BOUNDS_LO, N_CW), [1e-6]*Npulsars, RN_BOUNDS_LO, GWB_BOUNDS_LO])
    PARAM_HI = np.concatenate([np.tile(CW_BOUNDS_HI, N_CW), [30.0]*Npulsars, RN_BOUNDS_HI, GWB_BOUNDS_HI])
else:
    PARAM_LO = np.concatenate([np.tile(CW_BOUNDS_LO, N_CW), [1e-6]*Npulsars])
    PARAM_HI = np.concatenate([np.tile(CW_BOUNDS_HI, N_CW), [30.0]*Npulsars])

In [ ]:
# =============================================================================
# LOG-POSTERIOR (WITH MAPPING FUNCTIONS)

In [ ]:
# =============================================================================
def make_params_dict(x):
    d = dict(FIXED_PARAMS)
    for i, name in enumerate(SAMPLED_NAMES):
        d[name] = x[i]
    return d

@jax.jit
def logp(x):
    # Hard prior: reject if any parameter is outside its prior bounds
    in_bounds = jnp.all(x >= jnp.array(PARAM_LO)) & jnp.all(x <= jnp.array(PARAM_HI))
    params = dict(FIXED_PARAMS)
    for i, name in enumerate(SAMPLED_NAMES):
        params[name] = x[i]
        # Woodbury-marginalized log-likelihood (marginalizes over timing model, red noise, and GWB GP coefficients)
    ll = logl_disco(params)
    p_dists = x[N_CW_PARAMS:N_CW_PARAMS+Npulsars]
        # Gaussian EM distance prior: penalizes distances far from the EM measurement
    log_prior = -0.5 * jnp.sum(jnp.square((p_dists - dist_mu_jnp) / sd_jnp))
    result = jnp.where(in_bounds, ll + log_prior, -1e30)
        # NaN guard: extreme parameter combos can overflow the Woodbury computation
    return jnp.where(jnp.isnan(result), -1e30, result)

In [ ]:
# =============================================================================
# TRUTH VECTOR

In [ ]:
# =============================================================================
truth_vals = []
for name in SAMPLED_NAMES:
    if name in inj_cw_vals:
        truth_vals.append(inj_cw_vals[name])
    elif name in enterprise_params:
        truth_vals.append(enterprise_params[name])
    else:
        raise ValueError(f"No injection value for '{name}'")
truth = np.array(truth_vals)

print("\nTruth vector:")
for i, (name, val) in enumerate(zip(SAMPLED_NAMES, truth)):
    print(f"  [{i:2d}] {name:40s} = {val:.6f}")

print("\nCompiling logp...")
lp_truth = float(logp(jnp.array(truth)))
print(f"logp(truth) = {lp_truth:.4f}")

In [ ]:
# =============================================================================
# DISTANCE FRINGE SPACING (ALL SOURCES)

In [ ]:
# =============================================================================
@jax.jit
def compute_delta_L_single(cos_gwtheta, gwphi, log10_fgw):
    gwtheta = jnp.arccos(cos_gwtheta)
    f_gw    = 10.0 ** log10_fgw
    _, _, cos_mu = jax.vmap(lambda pos: fpcmu_fast(pos, gwtheta, gwphi))(psr_positions)
    denom = jnp.maximum(jnp.abs(1.0 - cos_mu), 1e-4)
    return 1.0 / (f_gw * KPC_OVER_C * denom)

def compute_min_delta_L(x):
    """
    Compute minimum fringe spacing across all CW sources for each pulsar. The minimum matters because the finest fringe sets the required distance proposal resolution.
    """
    all_dL = []
    for s in range(N_CW):
        idx = s*8
        dL_s = np.array(compute_delta_L_single(x[idx+0], x[idx+1], x[idx+4]))
        all_dL.append(dL_s)
    return np.min(all_dL, axis=0)

dL = compute_min_delta_L(truth)
print("\nDistance fringe spacings and modes per sigma (min over sources):")
for i, psr in enumerate(disco_psrs):
    print(f"  {psr.name}: d_true={truth[N_CW_PARAMS+i]:.4f} kpc, dL={dL[i]:.6f} kpc, modes/sig={sd_arr[i]/dL[i]:.0f}")

In [ ]:
# =============================================================================
# HESSIAN/FISHER MATRIX

In [ ]:
# =============================================================================
grad_logp = jax.jit(jax.grad(logp))
batch_logp = jax.jit(jax.vmap(logp))

print("Computing Hessian at truth...")
t0 = time.time()
H_full = np.array(jax.hessian(logp)(jnp.array(truth)))
print(f"Hessian computed in {time.time()-t0:.1f}s")
H_cw = H_full[:N_CW_PARAMS, :N_CW_PARAMS]
eig_cw, evec_cw = np.linalg.eigh(-H_cw)
eig_cw_c = np.maximum(eig_cw, 1e-12 * eig_cw.max())
cov_fisher   = evec_cw @ np.diag(1.0 / eig_cw_c) @ evec_cw.T
cov_fisher   = 0.5 * (cov_fisher + cov_fisher.T)
eig_vals_cov, eig_vecs_cov = np.linalg.eigh(cov_fisher)
eig_sigs_fisher = 2.38 * np.sqrt(np.maximum(eig_vals_cov, 1e-30))
fisher_sig_cw = np.sqrt(np.diag(cov_fisher))
# Distance Hessian diagonal
H_dist_diag = np.array([H_full[N_CW_PARAMS+j, N_CW_PARAMS+j] for j in range(Npulsars)])

print(f"Fisher eigenmode widths: min={eig_sigs_fisher.min():.2e}, max={eig_sigs_fisher.max():.2e}")

print("Pre-compiling batch_logp...")
_ = batch_logp(jnp.tile(jnp.array(truth), (40, 1)))
print("Done.")

In [ ]:
# =============================================================================
# STARTING POINT: Offset by 3-5 Fisher sigmas/random

In [ ]:
# =============================================================================
rng = np.random.default_rng(99)
x0  = truth.copy()
for i in range(N_CW_PARAMS):
    offset_sigma = rng.uniform(3.0, 5.0) * rng.choice([-1, 1])
    proposed     = x0[i] + offset_sigma * fisher_sig_cw[i]
    lo = PARAM_LO[i] + 1e-4
    hi = PARAM_HI[i] - 1e-4
    margin = 0.3 * (hi - lo)
    x0[i] = np.clip(proposed, lo + margin, hi - margin)
for j in range(Npulsars):
    offset   = rng.uniform(3.0, 5.0) * rng.choice([-1, 1]) * sd_arr[j]
    x0[N_CW_PARAMS+j] = max(truth[N_CW_PARAMS+j] + offset, 0.01)
if SAMPLE_NOISE:
    for j in range(N_CW_PARAMS + Npulsars, Ndim):
        x0[j] = truth[j] + rng.uniform(-0.5, 0.5)
        x0[j] = np.clip(x0[j], PARAM_LO[j] + 1e-4, PARAM_HI[j] - 1e-4)
lp_start = float(logp(jnp.array(x0)))
print(f"\nlogp(start) = {lp_start:.2f}")
print(f"logp(truth) = {lp_truth:.4f}")
print(f"Gap: {lp_truth - lp_start:.0f} nats")

In [ ]:
# =============================================================================
# SAMPLER UTILITIES (Newton snap, bounds)

In [ ]:
# =============================================================================
def snap_distances(x_prop, n_newton=3):
    """
    Newton-snap pulsar distances to their conditional MAP given current CW params. Uses the gradient and diagonal Hessian curvature to take Newton steps. This is critical because distance has ~100s of fringe modes per EM sigma — without snapping, random distance proposals almost never land on a good fringe.
    """
    for _ in range(n_newton):
        g = np.array(grad_logp(jnp.array(x_prop)))
        for j in range(Npulsars):
            idx = N_CW_PARAMS + j
            if H_dist_diag[j] < -1e-6:
                x_prop[idx] = max(x_prop[idx] - g[idx] / H_dist_diag[j], 1e-6)
    return x_prop

def in_bounds(x_prop):
    return np.all(x_prop >= PARAM_LO) and np.all(x_prop <= PARAM_HI)

In [ ]:
# =============================================================================
# MCMC SAMPLER: 3-phase Annealing/Adapt/Prod

In [ ]:
# =============================================================================
# Exponential cooling schedule: T decreases geometrically each step
cool_rate = (T_end / T_start) ** (1.0 / n_anneal)
n_total   = n_anneal + n_adapt + n_prod

print(f"\nAnnealing: T {T_start} -> {T_end} over {n_anneal} steps (cool_rate={cool_rate:.6f})")
print(f"Adapt:     {n_adapt} steps at T=1 to build empirical covariance")
print(f"Prod:      {n_prod} steps")

all_chain = np.zeros((n_total, Ndim))
all_lps   = np.zeros(n_total)
all_temps = np.zeros(n_total)
emp_samples = []
emp_cov     = None
use_emp_cov = False
scale_log = np.zeros(N_CW_PARAMS)
acc_counts = {'eigen': 0, 'dist': 0, 'joint': 0}
tot_counts = {'eigen': 0, 'dist': 0, 'joint': 0}

x  = x0.copy()
lp = float(logp(jnp.array(x)))
T  = T_start

t0 = time.time()
# Main MCMC loop: 3 phases
#   1. Annealing (steps 0..n_anneal): temperature cools from T_start to 1.0
#      High T flattens the posterior, allowing broad exploration and mode-hopping.
#   2. Adapt (n_anneal..n_anneal+n_adapt): T=1, build empirical covariance from
#      the second half of annealing samples. This replaces the Fisher-based proposal
#      with one learned from the chain's actual shape.
#   3. Production (last n_prod steps): T=1, fixed proposals, samples saved for inference.
for step in range(n_total):
    if step < n_anneal:
        phase = 'anneal'
        T     = T_start * (cool_rate ** step)
    else:
        phase = 'adapt' if step < n_anneal + n_adapt else 'prod'
        T     = 1.0
        # At the annealing→adapt transition, build an empirical covariance matrix from the CW parameters of the late annealing samples.
    if step == n_anneal and len(emp_samples) > N_CW_PARAMS * 2:
        emp_arr = np.array(emp_samples)
        last_n  = max(N_CW_PARAMS * 3, len(emp_arr) // 3)
        emp_arr = emp_arr[-last_n:, :N_CW_PARAMS]
        if emp_arr.shape[0] > N_CW_PARAMS:
            raw_cov = np.cov(emp_arr.T)
            raw_cov = 0.5 * (raw_cov + raw_cov.T)
            eig_e, vec_e = np.linalg.eigh(raw_cov)
            eig_e = np.maximum(eig_e, 1e-12 * eig_e.max())
            emp_cov_mat = vec_e @ np.diag(eig_e) @ vec_e.T
            scale_nd = 2.38**2 / N_CW_PARAMS
            try:
                L_emp = np.linalg.cholesky(scale_nd * emp_cov_mat)
                emp_cov = {'L': L_emp, 'eig_sigs': 2.38 * np.sqrt(eig_e), 'vecs': vec_e}
                use_emp_cov = True
                print(f"  [step {step}] Switched to empirical covariance (built from {emp_arr.shape[0]} samples)")
            except np.linalg.LinAlgError:
                print(f"  [step {step}] Empirical Cholesky failed, keeping Fisher cov")
    r = rng.random()
        # --- EIGENMODE PROPOSAL (50% of steps) ---
    # Proposes along a single eigenvector of the CW covariance matrix.
    # Efficient because CW parameters are highly correlated (e.g., frequency-chirp mass).
    # After proposing new CW params, Newton-snap distances to their conditional MAP.
    if r < 0.50:
        if use_emp_cov:
            mode_idx = rng.integers(N_CW_PARAMS)
            z        = rng.standard_normal()
            sig      = emp_cov['eig_sigs'][mode_idx]
            x_prop   = x.copy()
            x_prop[:N_CW_PARAMS] += z * sig * emp_cov['vecs'][:, mode_idx]
        else:
            mode_idx   = rng.integers(N_CW_PARAMS)
            z          = rng.standard_normal()
            scaled_sig = eig_sigs_fisher[mode_idx] * np.sqrt(T) * np.exp(scale_log[mode_idx])
            x_prop     = x.copy()
            x_prop[:N_CW_PARAMS] += z * scaled_sig * eig_vecs_cov[:, mode_idx]
        x_prop = snap_distances(x_prop)
        if in_bounds(x_prop):
            lp_prop  = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            accepted  = np.log(rng.random() + 1e-300) < log_alpha
            if accepted:
                x = x_prop; lp = lp_prop
            acc_counts['eigen'] += int(accepted)
        tot_counts['eigen'] += 1
                # Robbins-Monro step-size adaptation during annealing:
        # adjusts eigenmode scales toward 35% acceptance rate
        if phase == 'anneal':
            gamma = 1.0 / (step + 100)
            scale_log[mode_idx] += gamma * (float(x is x_prop) - 0.35)
            scale_log[mode_idx]  = np.clip(scale_log[mode_idx], -5.0, 10.0)
        # --- DISTANCE PROPOSAL (30% of steps) ---
    # Two-stage: (1) draw from EM prior (broadened by sqrt(T)),
    # (2) grid-scan one fringe period to find the best fringe mode.
    # Handles ~100s of fringe modes within each pulsar's EM sigma.
    elif r < 0.80:
        pi    = rng.integers(Npulsars)
        d_prop = rng.normal(mu_arr[pi], sd_arr[pi] * max(1.0, np.sqrt(T)))
        dL_j  = float(dL[pi])
        if d_prop > dL_j:
            x_snap = x.copy()
            x_snap[N_CW_PARAMS+pi] = d_prop
            d_lo    = max(d_prop - 0.6 * dL_j, 1e-6)
            d_hi    = d_prop + 0.6 * dL_j
            d_cands = np.linspace(d_lo, d_hi, 30)
            x_batch = np.tile(x_snap, (30, 1))
            x_batch[:, N_CW_PARAMS+pi] = d_cands
            lps_scan = np.array(batch_logp(jnp.array(x_batch)))
            x_prop = x.copy()
            x_prop[N_CW_PARAMS+pi] = float(d_cands[np.argmax(lps_scan)])
            lp_prop = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['dist'] += 1
        tot_counts['dist'] += 1
    else:
        # --- JOINT CW PROPOSAL (20% of steps) ---
        # Proposes all CW parameters simultaneously from the full covariance.
        # Low acceptance expected in high dimensions but can find better modes.
        if use_emp_cov:
            z      = rng.standard_normal(N_CW_PARAMS)
            x_prop = x.copy()
            x_prop[:N_CW_PARAMS] += emp_cov['L'] @ z
        else:
            scale_nd = 2.38**2 / N_CW_PARAMS
            L_fish   = np.linalg.cholesky(scale_nd * T * cov_fisher)
            z        = rng.standard_normal(N_CW_PARAMS)
            x_prop   = x.copy()
            x_prop[:N_CW_PARAMS] += L_fish @ z
        if in_bounds(x_prop):
            lp_prop   = float(logp(jnp.array(x_prop)))
            log_alpha = (lp_prop - lp) / T
            if np.log(rng.random() + 1e-300) < log_alpha:
                x = x_prop; lp = lp_prop
                acc_counts['joint'] += 1
        tot_counts['joint'] += 1
    all_chain[step] = x
    all_lps[step]   = lp
    all_temps[step] = T
    # Collect samples from second half of annealing for empirical covariance
    if phase == 'anneal' and step > n_anneal // 2:
        emp_samples.append(x.copy())
    if step % 2000 == 0:
        elapsed = time.time() - t0
        print(f"  step {step:5d}/{n_total} [{phase:6s}] T={T:7.2f} | logp={lp:10.2f} | "
              f"eigen={acc_counts['eigen']}/{tot_counts['eigen']} "
              f"dist={acc_counts['dist']}/{tot_counts['dist']} "
              f"joint={acc_counts['joint']}/{tot_counts['joint']} | {elapsed:.0f}s")

In [ ]:
# =============================================================================
# SUMMARY AND DIAGNOSTIC PLOTS

In [ ]:
# =============================================================================
dt = time.time() - t0
print(f"\nDone in {dt:.1f}s ({n_total/dt:.0f} it/s)")
for k in acc_counts:
    ar = acc_counts[k] / max(tot_counts[k], 1)
    print(f"  {k:10s}: {acc_counts[k]:5d}/{tot_counts[k]:5d} = {ar:.3f}")

prod_chain = all_chain[n_anneal + n_adapt:]
prod_lps   = all_lps[n_anneal + n_adapt:]

print(f"\nlogp truth = {lp_truth:.4f}")
print(f"logp prod:  mean={np.mean(prod_lps):.2f}, std={np.std(prod_lps):.2f}, max={np.max(prod_lps):.2f}")

print("\nCW parameter recovery (production chain):")
for s in range(N_CW):
    print(f"  --- src{s} ---")
    for pidx, pname in enumerate(CW_PARAM_NAMES):
        col = s*8 + pidx
        med = np.median(prod_chain[:, col])
        std = np.std(prod_chain[:, col])
        bias = med - truth[col]
        print(f"    {pname:15s}: truth={truth[col]:+.4f}, median={med:+.4f}, std={std:.2e}, bias={bias:+.4f}")

print("\nDistance recovery (production chain):")
for j in range(Npulsars):
    med       = np.median(prod_chain[:, N_CW_PARAMS+j])
    err_modes = abs(med - truth[N_CW_PARAMS+j]) / dL[j]
    print(f"  {disco_psrs[j].name}: med={med:.4f}, truth={truth[N_CW_PARAMS+j]:.4f}, err={err_modes:.0f} fringe modes")

# Diagnostic plots
dist_colours = ['#1a3a5c', '#8b2500', '#2d5a27', '#6a3d9a', '#b15928']
colours_src = ['#1a3a5c', '#b5442d', '#2d7a3d']
src_labels = [f'src{s}' for s in range(N_CW)]
ann_end   = n_anneal
adap_end  = n_anneal + n_adapt
steps_arr = np.arange(n_total)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
noise_label = "noise_vary" if SAMPLE_NOISE else "noise_fixed"
fig.suptitle(
    f'N_CW={N_CW}, h=10^{log10_h}, {noise_label}: PulsarLikelihood + Annealing\n'
    f'Start: dlogp={lp_start-lp_truth:.0f} from truth | {n_anneal}+{n_adapt}+{n_prod} steps',
    fontsize=13)

def add_phase_lines(ax):
    ax.axvline(ann_end,  color='blue',   ls=':', lw=1.5, alpha=0.7)
    ax.axvline(adap_end, color='orange', ls=':', lw=1.5, alpha=0.7)

# Row 0, Col 0: logp trace
ax = axes[0, 0]
ax.plot(steps_arr, all_lps, color='#333', lw=0.3, alpha=0.8)
ax.axhline(lp_truth, color='r', ls='--', lw=1, label=f'truth ({lp_truth:.2f})')
add_phase_lines(ax)
ax.set_xlabel('step'); ax.set_ylabel('logp')
ax.set_title('Log-posterior trace'); ax.legend(fontsize=7)

# Row 0, Col 1: temperature schedule
ax = axes[0, 1]
ax.semilogy(steps_arr[:n_anneal], all_temps[:n_anneal], color='#b5442d', lw=0.6)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_xlabel('step'); ax.set_ylabel('Temperature T')
ax.set_title('Temperature schedule (annealing)')

# Row 0, Col 2: log10_fgw for all sources
ax = axes[0, 2]
for s in range(N_CW):
    col = s*8 + 4
    ax.plot(steps_arr, all_chain[:, col], color=colours_src[s], lw=0.3, alpha=0.7, label=src_labels[s])
    ax.axhline(truth[col], color=colours_src[s], ls='--', lw=1)
add_phase_lines(ax)
ax.set_title('log10_fgw'); ax.set_xlabel('step'); ax.legend(fontsize=7)

# Row 1: cos_gwtheta, log10_h, cos_inc for all sources
params_show = [(0, 'cos_gwtheta'), (5, 'log10_h'), (2, 'cos_inc')]
for pidx, (param_off, param_name) in enumerate(params_show):
    ax = axes[1, pidx]
    for s in range(N_CW):
        col = s*8 + param_off
        ax.plot(steps_arr, all_chain[:, col], color=colours_src[s], lw=0.3, alpha=0.7, label=src_labels[s])
        ax.axhline(truth[col], color=colours_src[s], ls='--', lw=1)
        ax.plot(0, x0[col], 'o', color=colours_src[s], ms=5, zorder=5)
    add_phase_lines(ax)
    ax.set_title(param_name); ax.set_xlabel('step'); ax.legend(fontsize=7)

# Rows 2-3: distance traces
for j in range(min(Npulsars, 5)):
    row = 2 if j < 3 else 3
    col = j if j < 3 else j - 3
    ax = axes[row, col]
    ax.plot(steps_arr, all_chain[:, N_CW_PARAMS+j], color=dist_colours[j % 5], lw=0.3, alpha=0.8)
    ax.axhline(truth[N_CW_PARAMS+j], color='r', ls='--', lw=1, label='truth')
    ax.plot(0, x0[N_CW_PARAMS+j], 'o', color='orange', ms=5, zorder=5, label='start')
    add_phase_lines(ax)
    ax.set_title(f'{disco_psrs[j].name} dist'); ax.set_xlabel('step')
    ax.legend(fontsize=7)

# Row 3, Col 2: log10_h posterior for all sources
ax = axes[3, 2]
for s in range(N_CW):
    col = s*8 + 5
    ax.hist(prod_chain[:, col], bins=50, density=True, alpha=0.5,
            color=colours_src[s], label=f'src{s} (truth={truth[col]:.1f})')
    ax.axvline(truth[col], color=colours_src[s], ls='--', lw=1.5)
ax.set_title('log10_h posteriors (prod)'); ax.set_xlabel('log10_h'); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()